# Phase 4 — Training Pair Generation
**FSQA** | fine-tune pipeline

> **Bring your own data.** No data ships with this repo. This notebook reads `./local-data/phase1_fsqa_eval/<run id>/questions.jsonl` (JSON Lines, one question per line, as Phase 1 writes it) for the contamination guard. It uses the last run folder in name order. Run Phase 1 first, or supply your own file in that format. Each record needs `question`; it is the only field this notebook reads from that file.

Generates the training corpus for the fine-tune LoRA.

**Design**
- Generator: `GENERATOR` in the config cell; set it to the teacher you picked in Phases 2b, 2c and 3
- Scope: original 47 sections (21 CFR 1 Subpart L, Part 7, Part 810, Part 117)
  plus 33 verified sections from Parts 112, 123, and 120
- Scale: seed queue cycles section × archetype; pass > 0 adds a scenario-
  variation note so the model writes fresh scenarios, not rephrasings
- Target: 9,200 gross; the net count depends on the keep rate of your run
- Long sections (>7,500 chars) are chunked at subsection boundaries rather
  than cut at the 15k cap Phase 3 uses (Phase 3 reports scores by section,
  so you can check how long sections fare)
- Dedup: Jaccard 0.55 on content words, per-section, against the Phase 1 eval
  questions (contamination guard) then internally
- QA filter: every pair, 2 judges (Gemini/GCP + Kimi/DO);
  drop rule: both judges flag contradiction OR any answer_accuracy ≤ 2 OR
  any question_quality = 0; single contradiction flag -> review file
- Outputs: training_pairs.jsonl (raw) + training_pairs_chat.jsonl (fine-tune)
- Checkpointed at both generation and QA stages

In [ ]:
import concurrent.futures
import hashlib
import itertools
import json
import os
import random
import re
import statistics
import threading
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone

import requests

print("imports ok", flush=True)

## Config

In [ ]:
GENERATOR = {"model": "openai/gpt-5.5", "effort": "xhigh"}
EVAL_PROVIDER = {"order": ["openai"], "allow_fallbacks": False}

JUDGES = {
    "gemini": {"slug": "google/gemini-3.1-pro-preview",
               "provider": {"order": ["google-vertex", "google-ai-studio"],
                            "allow_fallbacks": True}},
    "kimi":   {"slug": "moonshotai/kimi-k3",
               "provider": {"order": ["digitalocean"], "allow_fallbacks": False}},
}

TARGET_GROSS   = 9200
CHUNK_CHARS    = 7500
DEDUP_JACCARD  = 0.55
GEN_WORKERS    = 10
QA_WORKERS     = 32
REQUEST_TIMEOUT = 600
MAX_RETRIES     = 4
MAX_TOKENS      = 32000
SEED = 20260825

VOLUME_MOUNT     = "./local-data"
QUESTIONS_SUBDIR = "phase1_fsqa_eval"
OUTPUT_SUBDIR    = "phase4_training_pairs"

ECFR_BASE = "https://www.ecfr.gov/api/versioner/v1/full"
ECFR_DATE = "2026-08-01"
ECFR_HDRS = {"User-Agent": "labs-benchmark/1.0 (research; you@example.com)"}

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY missing. Attach 'your-modal-secret'.")
if not os.path.isdir(VOLUME_MOUNT):
    raise RuntimeError(f"{VOLUME_MOUNT} not mounted.")

HEADERS = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json",
           "X-OpenRouter-Title": "labs-benchmark"}
random.seed(SEED)

# --- Spend guard -----------------------------------------------------------
# Hard ceiling for the run, in USD. Every OpenRouter attempt is reserved at its worst case before it is sent and
# settled to its actual cost after. An attempt that would cross this is not sent. 25.00 is the default cap per run.
import math


# --- Canonical env parsing and ledger-root rule (identical in every notebook) ---------------------------------
# RUN_BUDGET_USD: unset means the default. Any value that is set, a blank or whitespace
# value included, must parse as a finite float greater than 0, or the notebook refuses to start.
def _env_usd(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return float(default)
    try:
        v = float(raw.strip())
    except ValueError:
        v = float("nan")
    if not math.isfinite(v) or v <= 0:
        raise RuntimeError(f"Refusing to start: {name} must be a finite number greater than 0 (got {raw!r}).")
    return v


RUN_BUDGET_USD = _env_usd("RUN_BUDGET_USD", 25.00)   # one cap per run; set the RUN_BUDGET_USD environment variable to change it
RUN_ID = None              # None = new run with a fresh ledger. Set a non-empty string to resume and keep its spend.
# Confirmed OpenRouter list prices, $/M tokens (input, output), retrieved 2026-11-05. Each is the highest price across
# the model's endpoints and tiers, so the reservation is a worst case. Sources:
#   https://openrouter.ai/api/v1/models/openai/gpt-5.5/endpoints (OpenAI Fast tier, prompts over 272k tokens)
#   https://openrouter.ai/api/v1/models/google/gemini-3.1-pro-preview/endpoints (priority tier, prompts over 200k tokens)
#   https://openrouter.ai/api/v1/models/moonshotai/kimi-k3/endpoints (priority endpoints; provider routing sets the real price)
# Every model that is called needs an entry here. A model with no confirmed price is left out, and the notebook then
# refuses to run before any paid call.
PRICE = {
    "openai/gpt-5.5": (25.00, 112.50),
    "google/gemini-3.1-pro-preview": (7.20, 32.40),
    "moonshotai/kimi-k3": (4.50, 22.50),
}

# A generator or judge model without a confirmed price cannot be bounded, so the run refuses to start.
missing_prices = sorted(({GENERATOR["model"]} | {j["slug"] for j in JUDGES.values()}) - set(PRICE))
if missing_prices:
    raise RuntimeError(f"No confirmed price for model(s) {missing_prices}. Nothing was sent. Add the listed USD per "
                       "million tokens (input, output) for each model to PRICE in this config cell, then rerun.")
GEN_PRICE = PRICE[GENERATOR["model"]]  # $/M input, output; used only for the cost estimate in the generation summary
# None is the only signal for a new run. Any other value is a resume ID and goes through _checked_run_id below,
# which refuses an empty string, 0, False and every other non-string value. Nothing is defaulted.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
def _checked_run_id(value):
    """RUN_ID names one directory under SPEND_ROOT. Refuse an absolute path or any '..' segment, so the text of
    RUN_ID cannot lead os.path.join(SPEND_ROOT, RUN_ID) outside SPEND_ROOT. This is a string check only: it does
    not detect a symlink that already exists under SPEND_ROOT. Fails loudly."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Refusing to start: RUN_ID must be a non-empty string (got {value!r}).")
    if os.path.isabs(value) or value.startswith(("/", "\\")) or os.path.splitdrive(value)[0]:
        raise RuntimeError(f"Refusing to start: RUN_ID must be a relative name, not an absolute path (got {value!r}).")
    if ".." in value.replace("\\", "/").split("/"):
        raise RuntimeError(f"Refusing to start: RUN_ID must not contain a '..' segment (got {value!r}).")
    return value
RUN_ID = _checked_run_id(RUN_ID)
RUN_DIR = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)


# One ledger root and lock root for every notebook: SPEND_LEDGER_DIR, default <repo root>/spend_ledger.
# Ledger root. If SPEND_LEDGER_DIR is set at all, it is stripped: blank or whitespace-only refuses, anything else
# becomes os.path.abspath(os.path.expanduser(value)). If it is not set, the root is <repo root>/spend_ledger, where
# the repo root is the first directory, starting at the absolute working directory and walking up, for which
# os.path.exists(<dir>/.git) is true (a .git directory or a .git file). With no such directory the notebook refuses:
# there is no fallback to the working directory, because a different working directory would give a different lock.
# The lock is <root>/runner.lock and the run ledgers are <root>/<RUN_ID>/spend_ledger.jsonl.
def _ledger_root():
    raw = os.environ.get("SPEND_LEDGER_DIR")
    if raw is not None:
        value = raw.strip()
        if not value:
            raise RuntimeError("Refusing to start: SPEND_LEDGER_DIR is set but blank. Unset it to use "
                               "<repo root>/spend_ledger, or set it to a directory.")
        return os.path.abspath(os.path.expanduser(value))
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, ".git")):
            return os.path.join(d, "spend_ledger")
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError(f"Refusing to start: SPEND_LEDGER_DIR is not set and there is no .git above the working "
                               f"directory ({start}). Set SPEND_LEDGER_DIR to the shared ledger directory.")
        d = parent


SPEND_LEDGER_ROOT = _ledger_root()
LEDGER_DIR = os.path.join(SPEND_LEDGER_ROOT, RUN_ID)
os.makedirs(LEDGER_DIR, exist_ok=True)
LEDGER_PATH = os.path.join(LEDGER_DIR, "spend_ledger.jsonl")

# One paid runner at a time across all notebooks. A single exclusive lock file in the shared ledger root: a second
# runner refuses to start while it exists. It is removed on a clean exit only. A stale lock (a runner that died
# without exiting) is never cleared automatically: confirm no runner is alive, then delete the runner.lock file
# named in the error by hand to resume.
import atexit
RUNNER_LOCK = os.path.join(SPEND_LEDGER_ROOT, "runner.lock")
try:
    _lock_fd = os.open(RUNNER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
except FileExistsError:
    raise RuntimeError(f"{RUNNER_LOCK} exists. Another runner may be using this run, or a runner stopped without "
                       "releasing it. Refusing to start. Check for a live runner before deleting the lock by hand.")
_RUNNER_TOKEN = f"pid {os.getpid()} {datetime.now(timezone.utc).isoformat()}"
os.write(_lock_fd, _RUNNER_TOKEN.encode())
os.close(_lock_fd)


def _release_runner_lock():
    try:
        with open(RUNNER_LOCK) as f:
            if f.read() == _RUNNER_TOKEN:
                os.remove(RUNNER_LOCK)
    except OSError:
        pass


atexit.register(_release_runner_lock)


print(f"Generator    : {GENERATOR}")
print(f"Target gross : {TARGET_GROSS}")
print(f"Chunk cap    : {CHUNK_CHARS} chars")
print(f"Budget       : ${RUN_BUDGET_USD:.2f} per run   ledger: {LEDGER_PATH}")

## CFR section lists

In [ ]:
ORIGINAL_SECTIONS = {
    "fsvp":   ["1.500","1.501","1.502","1.503","1.504","1.505","1.506","1.507",
                "1.508","1.509","1.510","1.511","1.512","1.513","1.514"],
    "recall": ["7.40","7.41","7.42","7.45","7.46","7.49","7.50","7.53","7.55","7.59",
               "810.10","810.11","810.12","810.13","810.15","810.16","810.17","810.18"],
    "pchf":   ["117.3","117.5","117.126","117.130","117.135","117.139","117.140",
               "117.150","117.155","117.160","117.165","117.170","117.180","117.190"],
}

EXPANDED_SECTIONS = {
    # Part 112 — Produce Safety (FSMA). Verified against eCFR 2026-08-01.
    "produce": ["112.1","112.2","112.3","112.11","112.12","112.30","112.31",
                "112.43","112.44","112.50","112.53","112.54",
                "112.81","112.111","112.112","112.140","112.141","112.143"],
    # Part 123 — Seafood HACCP
    "seafood": ["123.3","123.6","123.7","123.8","123.9","123.10","123.11","123.12"],
    # Part 120 — Juice HACCP
    "juice":   ["120.3","120.6","120.7","120.8","120.11","120.12","120.14"],
}

ARCHETYPES = [
    "applicability",
    "timeline or deadline",
    "recordkeeping requirement",
    "exemption or qualified exemption",
    "definition",
    "procedural sequence",
    "numeric threshold",
    "responsible party",
    "consequence of non-compliance",
    "conflict between two provisions",
]

VARIATION_NOTES = [
    "Focus on a small business (fewer than 500 employees) importing from Latin America.",
    "Focus on a large multi-site manufacturer importing from Southeast Asia.",
    "Focus on a produce distributor sourcing from small family farms.",
    "Focus on a cooperative of small seafood processors supplying a national retailer.",
    "Focus on a private-label importer who does not manufacture the food.",
]

ALL_SECTION_MAP = {}
for domain, secs in {**ORIGINAL_SECTIONS, **EXPANDED_SECTIONS}.items():
    for s in secs:
        ALL_SECTION_MAP[s] = domain

print("Section inventory:")
for d, secs in {**ORIGINAL_SECTIONS, **EXPANDED_SECTIONS}.items():
    print(f"  {d:<8} {len(secs):>3}")
print(f"  TOTAL    {len(ALL_SECTION_MAP):>3}")

## eCFR fetch and chunking

In [ ]:
def fetch_cfr_section(section, retries=4):
    part = section.split(".")[0]
    url  = f"{ECFR_BASE}/{ECFR_DATE}/title-21.xml"
    for attempt in range(retries):
        try:
            r = requests.get(url, params={"part": part, "section": section},
                             timeout=60, headers=ECFR_HDRS)
            if r.status_code == 200:
                r.encoding = "utf-8"
                txt = re.sub(r"<[^>]+>", " ", r.text)
                txt = re.sub(r"\s+", " ", txt).strip()
                m = re.match(r"§\s*[\d.]+\s+(.+?\.)\s", txt)
                heading = m.group(1).strip() if m else f"Section {section}"
                return heading, txt
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** (attempt + 2))
                continue
            return None, None
        except Exception:
            time.sleep(2 ** attempt)
    return None, None


def chunk_text(section, heading, text, cap=CHUNK_CHARS):
    if len(text) <= cap:
        return [(f"{section} — {heading}", text)]
    segs = re.split(r"(?=\((?:[a-z]|[ivx]+)\)\s)", text)
    chunks, cur = [], ""
    for seg in segs:
        if len(cur) + len(seg) > cap and cur:
            chunks.append(cur.strip()); cur = seg
        else:
            cur += seg
    if cur.strip():
        chunks.append(cur.strip())
    if not chunks:
        return [(f"{section} — {heading}", text[:cap])]
    return [(f"{section} chunk {i+1}/{len(chunks)} — {heading}", c)
            for i, c in enumerate(chunks)]


def q_count(n):
    return 8 if n < 1000 else 12 if n < 2500 else 16 if n < 5000 else 20


print(f"Fetching {len(ALL_SECTION_MAP)} CFR sections from eCFR...", flush=True)
CFR = {}
for i, (sec, domain) in enumerate(ALL_SECTION_MAP.items(), 1):
    print(f"  [{i:>2}/{len(ALL_SECTION_MAP)}] {sec}", flush=True)
    h, t = fetch_cfr_section(sec)
    if not t:
        print(f"    SKIP")
        continue
    CFR[sec] = [{"label": lbl, "text": chunk, "chars": len(chunk),
                 "max_q": q_count(len(chunk)), "domain": domain, "section": sec}
                for lbl, chunk in chunk_text(sec, h, t)]
    time.sleep(0.25)

total_sections = len(CFR)
total_chunks   = sum(len(v) for v in CFR.values())
chunked = {s for s, v in CFR.items() if len(v) > 1}
print(f"\nRetrieved: {total_sections}/{len(ALL_SECTION_MAP)} sections, {total_chunks} chunks")
print(f"Chunked  : {sorted(chunked)}")

## Load eval questions (contamination guard)

In [ ]:
def list_runs(subdir, req):
    base = os.path.join(VOLUME_MOUNT, subdir)
    return sorted(d for d in os.listdir(base)
                  if not d.startswith("_")
                  and os.path.isfile(os.path.join(base, d, req)))

rid = list_runs(QUESTIONS_SUBDIR, "questions.jsonl")[-1]
EVAL_Q = [json.loads(l) for l in open(
    os.path.join(VOLUME_MOUNT, QUESTIONS_SUBDIR, rid, "questions.jsonl"))]
print(f"Phase 1 run: {rid}  ({len(EVAL_Q)} eval questions)")

STOP = set("the a an of to in for and or is are be if what when must may you "
           "your this that with from on at as it its under section cfr does do "
           "can shall will not no any all each".split())

def cw(s):
    return frozenset(w for w in re.findall(r"[a-z]{4,}", s.lower()) if w not in STOP)

EVAL_FP = [cw(q["question"]) for q in EVAL_Q]

def too_similar_to_eval(question, t=DEDUP_JACCARD):
    c = cw(question)
    return any(len(c & fp) / len(c | fp) >= t for fp in EVAL_FP if c | fp)

print(f"Eval fingerprints: {len(EVAL_FP)}")

## Seed queue

In [ ]:
def build_seeds(target):
    all_chunks = [(sec, chunk) for sec, chunks in CFR.items() for chunk in chunks]
    seeds, total, pass_idx = [], 0, 0
    seen = set()
    a_it  = itertools.cycle(range(len(ARCHETYPES)))
    c_it  = itertools.cycle(range(len(all_chunks)))
    guard = 0
    while total < target:
        guard += 1
        if guard > target * 50:
            print("WARNING: seed loop guard hit"); break
        ci  = next(c_it)
        sec, chunk = all_chunks[ci]
        ai  = next(a_it)
        key = (sec, chunk["label"], ai, pass_idx)
        if key in seen:
            if ci == 0:
                pass_idx += 1; seen.clear()
            continue
        seen.add(key)
        uid = f"{chunk['domain'][:3]}-p{pass_idx}-{sec.replace('.','')}-c{ci%100:02d}-a{ai:02d}"
        var = VARIATION_NOTES[(pass_idx-1) % len(VARIATION_NOTES)] if pass_idx > 0 else ""
        seeds.append({"batch_id": uid, "section": sec, "domain": chunk["domain"],
                      "label": chunk["label"], "text": chunk["text"],
                      "max_q": chunk["max_q"], "archetype": ARCHETYPES[ai],
                      "pass_idx": pass_idx, "variation_note": var})
        total += chunk["max_q"]
    return seeds

SEEDS = build_seeds(TARGET_GROSS)
print(f"Seeds: {len(SEEDS)} batches, projected {sum(s['max_q'] for s in SEEDS):,} pairs")
print(f"Passes: 0..{max(s['pass_idx'] for s in SEEDS)}")
print(f"By domain: {dict(Counter(s['domain'] for s in SEEDS))}")

## Checkpoint

In [ ]:
class Checkpoint:
    def __init__(self, path, key_fn):
        self.path, self.key_fn = path, key_fn
        self.lock = threading.Lock()
        self.rows, self.done = [], set()
        if os.path.exists(path):
            for line in open(path):
                line = line.strip()
                if not line: continue
                try:
                    r = json.loads(line)
                except json.JSONDecodeError:
                    continue
                self.rows.append(r); self.done.add(key_fn(r))
        self.fh = open(path, "a")

    def has(self, k): return k in self.done

    def add(self, row):
        with self.lock:
            self.fh.write(json.dumps(row) + "\n")
            self.fh.flush(); os.fsync(self.fh.fileno())
            self.rows.append(row); self.done.add(self.key_fn(row))

    def close(self):
        try: self.fh.close()
        except Exception: pass

CKPT_DIR = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, "_checkpoint")
os.makedirs(CKPT_DIR, exist_ok=True)
gen_ckpt = Checkpoint(os.path.join(CKPT_DIR, f"gen_{rid}.jsonl"),
                      lambda r: r["batch_id"])
qa_ckpt  = Checkpoint(os.path.join(CKPT_DIR, f"qa_{rid}.jsonl"),
                      lambda r: (r["judge"], r["pair_id"]))
print(f"Gen ckpt: {len(gen_ckpt.rows)} batches | QA ckpt: {len(qa_ckpt.rows)} gradings")

## Request helper

In [ ]:
import threading, uuid
from decimal import Decimal


class BudgetExceeded(RuntimeError):
    pass


PROMPT_OVERHEAD_TOKENS = 256    # allowance for chat-template tokens that the byte bound below does not cover


def max_price_for(model):
    """provider.max_price for one model: its listed price in USD per million tokens (input, output). PRICE already
    holds per-million figures, so the value is converted with Decimal and not scaled. An unpriced model is refused."""
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    pin, pout = PRICE[model]
    return {"prompt": float(Decimal(str(pin))), "completion": float(Decimal(str(pout)))}


def openrouter_body(body):
    """The body every OpenRouter call sends: the caller's body with provider.max_price set from PRICE and usage
    accounting switched on. Done here, once, so no call site can leave either out."""
    b = dict(body)
    b["provider"] = {**(body.get("provider") or {}), "max_price": max_price_for(body["model"])}
    b["usage"] = {"include": True}
    return b


def worst_case_cost(body):
    """Upper bound for one attempt. Input comes from the UTF-8 byte count of the messages (each token covers
    at least one byte); output is capped by max_tokens, which includes reasoning. Fails closed."""
    model = body["model"]
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    if not isinstance(body.get("max_tokens"), int):
        raise BudgetExceeded(f"not sent: request for {model} has no max_tokens, so its cost cannot be bounded.")
    pin, pout = PRICE[model]
    in_bound = len(json.dumps(body["messages"], ensure_ascii=False).encode("utf-8")) + PROMPT_OVERHEAD_TOKENS
    return pin / 1e6 * in_bound + pout / 1e6 * body["max_tokens"]


def _is_count(x):
    return isinstance(x, int) and not isinstance(x, bool) and x >= 0


def attempt_cost(body, usage):
    """Settle only against complete, validated provider usage: a reported cost that is finite and above zero,
    and both token counts as non-negative integers. Anything less keeps the worst case that was reserved, so a
    zero, missing, partial or malformed reply cannot release the reservation."""
    if isinstance(usage, dict):
        c = usage.get("cost")
        if (isinstance(c, (int, float)) and not isinstance(c, bool) and 0 < c < float("inf")
                and _is_count(usage.get("prompt_tokens")) and _is_count(usage.get("completion_tokens"))):
            return float(c), "usage"
    return worst_case_cost(body), "worst_case"


_ledger_lock = threading.Lock()
_ledger = {}                    # attempt id -> entry; reloaded from LEDGER_PATH on resume
_budget_halted = False          # set by a refused dispatch or by an overrun; no later attempt is sent
if os.path.exists(LEDGER_PATH):
    with open(LEDGER_PATH) as _f:
        for _line in _f:
            if _line.strip():
                _rec = json.loads(_line)
                _ledger.setdefault(_rec["id"], {}).update(_rec)
    _budget_halted = any(_e.get("overrun") for _e in _ledger.values())   # an overrun halts the run after a restart too


def ledger_total():             # caller holds _ledger_lock. A reserved attempt with no settled line counts at worst case.
    return sum(e["cost"] if e["state"] == "settled" else e["worst_case"] for e in _ledger.values())


def _ledger_write(rec):
    with open(LEDGER_PATH, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def ledger_reserve(body, purpose):
    """Reserve one attempt at its worst case before it is sent. Raises BudgetExceeded if the run is halted, or if
    the reservation would take the run past RUN_BUDGET_USD; landing exactly on the cap is allowed."""
    global _budget_halted
    wc = round(worst_case_cost(body), 6)
    with _ledger_lock:
        if _budget_halted:
            raise BudgetExceeded(f"not sent: the run is halted, so {body['model']} ({purpose}) is refused before dispatch")
        if ledger_total() + wc <= RUN_BUDGET_USD:
            aid = uuid.uuid4().hex
            e = {"id": aid, "state": "reserved", "model": body["model"], "purpose": purpose,
                 "worst_case": wc, "ts": datetime.now(timezone.utc).isoformat()}
            _ledger[aid] = e
            _ledger_write(e)
            return aid
        _budget_halted = True
    raise BudgetExceeded(f"not sent: {body['model']} ({purpose}), worst case ${wc:.4f}, "
                         f"would take the run past RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}")


def ledger_settle(aid, body, usage):
    """Settle one attempt. A reported charge above the reservation for this attempt is an overrun: it is written to
    the ledger with its reservation and halts the run, so every later dispatch is refused before any request is sent."""
    global _budget_halted
    cost, source = attempt_cost(body, usage)
    with _ledger_lock:
        e = _ledger[aid]
        charged = round(cost, 6)
        overrun = source == "usage" and charged > e["worst_case"]
        e.update(state="settled", cost=charged, cost_source=source)
        rec = {"id": aid, "state": "settled", "cost": charged, "cost_source": source}
        if overrun:
            e["overrun"] = True
            rec.update(overrun=True, reserved=e["worst_case"])
            _budget_halted = True
        _ledger_write(rec)
    if overrun:
        print(f"Ledger overrun: an attempt was charged ${charged:.4f} against a reservation of ${e['worst_case']:.4f}. "
              f"The run is halted; no later attempt will be sent.")
    return e["cost"]


def ledger_summary():
    with _ledger_lock:
        es, total = list(_ledger.values()), ledger_total()
    by = {}
    for e in es:
        n, c = by.get(e["purpose"], (0, 0.0))
        by[e["purpose"]] = (n + 1, c + (e["cost"] if e["state"] == "settled" else e["worst_case"]))
    lines = [f"Spend ledger: {len(es)} attempts, RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}"]
    lines += [f"  {p:8s} attempts={n:5d}  charged=${c:.4f}" for p, (n, c) in sorted(by.items())]
    lines.append(f"  total charged ${total:.4f}   reserved, not settled={sum(e['state'] == 'reserved' for e in es)}")
    return "\n".join(lines)




def call_or(body, timeout=REQUEST_TIMEOUT, retries=MAX_RETRIES, empty_retry=False, purpose="call"):
    body = openrouter_body(body)            # provider.max_price and usage accounting on every request
    last = None
    for attempt in range(retries):
        aid = ledger_reserve(body, purpose)   # raises BudgetExceeded before anything is sent
        usage = None
        try:
            r = requests.post(OPENROUTER_URL, headers=HEADERS, json=body, timeout=timeout)
            if r.status_code in (429, 503):
                w = r.headers.get("Retry-After")
                time.sleep(float(w) if w and w.isdigit() else 2**(attempt+2))
                last = f"HTTP {r.status_code}"; continue
            if r.status_code >= 400:
                last = f"HTTP {r.status_code}: {r.text[:300]}"
                if 400 <= r.status_code < 500 and r.status_code != 429:
                    return {"ok": False, "error": last}
                time.sleep(2**attempt); continue
            d = r.json()
            usage = d.get("usage") or None
            if "error" in d:
                return {"ok": False, "error": str(d["error"])[:300]}
            text = (d["choices"][0]["message"].get("content") or "").strip()
            if not text and empty_retry and attempt < retries-1:
                time.sleep(1+attempt); continue
            u = d.get("usage", {}) or {}
            return {"ok": True, "text": text,
                    "prompt_tokens": u.get("prompt_tokens", 0),
                    "completion_tokens": u.get("completion_tokens", 0), "error": None}
        except Exception as e:
            last = f"{type(e).__name__}: {str(e)[:300]}"; time.sleep(2**attempt)
        finally:
            ledger_settle(aid, body, usage)
    return {"ok": False, "error": last}

print("helper loaded")

## Generation

In [ ]:
GEN_SYSTEM = """You are generating training data for a food safety regulatory AI. Each pair must teach the model to reason from regulation text to a precise, practitioner-relevant answer.

The full text of the governing CFR section is provided. Write questions and answers from that text only. Do not rely on memory."""

GEN_TEMPLATE = """Generate up to {n} question-and-answer pairs.

=== GOVERNING SECTION: {label} ===
{text}
=== END SECTION ===

ARCHETYPE: {archetype}
{variation_block}
Each question must:
- Be answerable from the section text above
- Be open-ended. No multiple choice, no yes/no
- Describe a real situation a food safety professional would face
- Use practitioner language, not regulatory boilerplate
- Be distinct from all others in this batch

Each reference answer must:
- Cite the specific subsection (e.g., "21 CFR {section}(b)(2)")
- Track the operative language from the text closely
- State the requirement in under 120 words
- Note conditions or exceptions that change the answer
- Contain nothing not in the section text above

Difficulty mix: 30% recall, 50% application, 20% reconciliation.

Return fewer if the section cannot support {n} distinct pairs of this archetype.
Do not pad. Do not invent subsections.

Return JSON only, matching the schema."""

GEN_SCHEMA = {
    "type": "json_schema",
    "json_schema": {"name": "training_batch", "strict": True,
        "schema": {"type": "object", "properties": {"pairs": {"type": "array", "items": {
            "type": "object",
            "properties": {"question": {"type": "string"},
                           "reference_answer": {"type": "string"},
                           "citation": {"type": "string"},
                           "difficulty": {"type": "string",
                                          "enum": ["recall","application","reconciliation"]}},
            "required": ["question","reference_answer","citation","difficulty"],
            "additionalProperties": False}}},
        "required": ["pairs"], "additionalProperties": False}},
}

CIT_RE = re.compile(r"21\s*CFR\s*\d+[\.\d]+(?:\([a-z0-9]+\))*")

def norm_cite(q):
    c = q.get("citation")
    if isinstance(c, str) and c.strip(): return c
    cs = q.get("citations")
    if isinstance(cs, list) and cs: return "; ".join(str(x) for x in cs)
    if isinstance(cs, str): return cs
    m = CIT_RE.search(q.get("reference_answer",""))
    return m.group(0) if m else ""

def generate_batch(seed):
    var = f"\nSCENARIO CONSTRAINT: {seed['variation_note']}\n" if seed["variation_note"] else ""
    body = {
        "model": GENERATOR["model"], "provider": EVAL_PROVIDER,
        "reasoning": {"effort": GENERATOR["effort"]},
        "max_tokens": MAX_TOKENS, "response_format": GEN_SCHEMA,
        "messages": [
            {"role": "system", "content": GEN_SYSTEM},
            {"role": "user", "content": GEN_TEMPLATE.format(
                n=seed["max_q"], label=seed["label"], text=seed["text"],
                archetype=seed["archetype"], variation_block=var,
                section=seed["section"])}],
    }
    res = call_or(body, purpose="gen")
    row = {"batch_id": seed["batch_id"], "section": seed["section"],
           "domain": seed["domain"], "archetype": seed["archetype"],
           "pass_idx": seed["pass_idx"], "asked": seed["max_q"],
           "ok": res["ok"], "error": res.get("error"),
           "prompt_tokens": res.get("prompt_tokens", 0),
           "completion_tokens": res.get("completion_tokens", 0), "pairs": []}
    if res["ok"]:
        try:
            txt = res["text"].replace("```json","").replace("```","").strip()
            b = txt.find("{"); txt = txt[b:] if b > 0 else txt
            raw = json.loads(txt).get("pairs", [])
            uid = hashlib.md5(seed["batch_id"].encode()).hexdigest()[:6]
            for i, p in enumerate(raw):
                cite = norm_cite(p)
                if not CIT_RE.search(cite) or seed["section"] not in cite: continue
                if len(p.get("question","")) < 20: continue
                row["pairs"].append({
                    "id": f"p4-{uid}-{i:02d}",
                    "question": p["question"],
                    "reference_answer": p["reference_answer"],
                    "citation": cite,
                    "difficulty": p.get("difficulty","application"),
                    "section": seed["section"], "domain": seed["domain"],
                    "batch_id": seed["batch_id"], "pass_idx": seed["pass_idx"],
                    "archetype": seed["archetype"],
                })
        except Exception as e:
            row["error"] = f"parse: {type(e).__name__}"
    gen_ckpt.add(row)
    return row

todo = [s for s in SEEDS if not gen_ckpt.has(s["batch_id"])]
run_id = RUN_ID
print(f"Run {run_id}")
print(f"  todo : {len(todo):,}  done: {len(SEEDS)-len(todo):,}", flush=True)

_t0 = time.time(); _n = [0]
def _tgen(s):
    r = generate_batch(s)
    _n[0] += 1
    if _n[0] % 50 == 0:
        el = time.time() - _t0
        done_p = sum(len(b.get("pairs",[])) for b in gen_ckpt.rows)
        print(f"  {_n[0]:,}/{len(todo):,} batches  {done_p:,} pairs  "
              f"eta {(len(todo)-_n[0])/(_n[0]/el)/60:.0f}m", flush=True)
    return r

if todo:
    with concurrent.futures.ThreadPoolExecutor(max_workers=GEN_WORKERS) as pool:
        list(pool.map(_tgen, todo))

gen_elapsed = time.time() - _t0
RAW = [p for b in gen_ckpt.rows for p in b.get("pairs",[])]
gen_cost = sum(b["prompt_tokens"]/1e6*GEN_PRICE[0]+b["completion_tokens"]/1e6*GEN_PRICE[1]
               for b in gen_ckpt.rows)
print(f"\nGeneration: {gen_elapsed/60:.1f} min | {len(RAW):,} raw pairs | ${gen_cost:.2f}")
print(f"  by domain: {dict(Counter(p['domain'] for p in RAW))}")

## Deduplication

In [ ]:
def dedup(pairs):
    seen, keep, n_ev, n_int = {}, [], 0, 0
    for p in pairs:
        c = cw(p["question"])
        if not c: keep.append(p); continue
        if too_similar_to_eval(p["question"]): n_ev += 1; continue
        sec = p["section"]
        if any(len(c & fp)/len(c|fp) >= DEDUP_JACCARD for fp in seen.get(sec,[]) if c|fp):
            n_int += 1; continue
        seen.setdefault(sec,[]).append(c)
        keep.append(p)
    return keep, n_ev, n_int

DEDUPED, n_ev, n_int = dedup(RAW)
print(f"Dedup: {len(RAW):,} -> {len(DEDUPED):,}")
print(f"  eval-contaminated: {n_ev} | internal dupes: {n_int}")

if len(DEDUPED) < 7000:
    print(f"WARNING: only {len(DEDUPED):,} pairs after dedup. "
          f"Consider raising TARGET_GROSS.")

## QA filter (every pair, 2 judges, checkpointed)

In [ ]:
QA_SYSTEM = """You review question-and-answer pairs for a food safety regulatory training dataset. You have the full text of the governing CFR section.

question_quality (0-3): Is the question well-formed and cleanly answerable from this section?
  3=realistic and specific  2=answerable but generic  1=vague/outside section  0=malformed

answer_accuracy (0-5): Is the answer correct as a statement of this section?
  5=correct and precise  4=minor imprecision  3=one meaningful gap  2=material error  1=mostly wrong  0=wrong

pair_coherence (0-2): Does the answer answer the question asked?
  2=directly  1=partially  0=no

citation_valid (0-2): Does the citation exist in the text above and support the answer?
  2=yes  1=exists but wrong subsection  0=does not exist in the text

contradiction (boolean): Answer asserts something the CFR text directly contradicts or a requirement not in the section. Citing a related provision elsewhere is NOT a contradiction.

Return JSON only:
{"question_quality": N, "answer_accuracy": N, "pair_coherence": N, "citation_valid": N, "contradiction": true|false}"""

QA_TEMPLATE = """=== CFR SECTION: {section} ===
{text}
=== END ===

QUESTION: {question}

REFERENCE ANSWER: {answer}

CITATION: {citation}

Grade. JSON only."""

def qa_one(args):
    jn, p = args
    j = JUDGES[jn]
    chunks = CFR.get(p["section"],[])
    text = chunks[0]["text"][:8000] if chunks else ""
    body = {
        "model": j["slug"], "provider": j["provider"], "max_tokens": 1000,
        "messages": [
            {"role": "system", "content": QA_SYSTEM},
            {"role": "user", "content": QA_TEMPLATE.format(
                section=p["section"], text=text,
                question=p["question"], answer=p["reference_answer"],
                citation=p["citation"])}],
    }
    res = call_or(body, timeout=180, empty_retry=True, purpose="qa")
    row = {"judge": jn, "pair_id": p["id"], "section": p["section"], "domain": p["domain"]}
    if not res["ok"] or not res["text"]:
        row.update({"answer_accuracy": None, "error": res.get("error") or "empty"})
    else:
        try:
            txt = res["text"].replace("```json","").replace("```","").strip()
            b = txt.find("{"); txt = txt[b:] if b > 0 else txt
            g = json.loads(txt)
            row.update({k: int(g.get(k,0))
                        for k in ["question_quality","answer_accuracy","pair_coherence","citation_valid"]})
            row["contradiction"] = bool(g.get("contradiction"))
            row["error"] = None
        except Exception as e:
            row.update({"answer_accuracy": None,
                        "error": f"parse: {type(e).__name__} :: {res['text'][:100]}"})
    qa_ckpt.add(row)
    return row

qa_tasks = [(jn, p) for jn in JUDGES for p in DEDUPED
            if not qa_ckpt.has((jn, p["id"]))]
print(f"QA: {len(JUDGES)*len(DEDUPED):,} total | {len(qa_tasks):,} remaining", flush=True)

_q0 = time.time(); _qn = [0]
def _tqa(t):
    r = qa_one(t)
    _qn[0] += 1
    if _qn[0] % 500 == 0:
        el = time.time() - _q0
        print(f"  QA {_qn[0]:,}/{len(qa_tasks):,}  "
              f"eta {(len(qa_tasks)-_qn[0])/(_qn[0]/el)/60:.0f}m", flush=True)
    return r

if qa_tasks:
    with concurrent.futures.ThreadPoolExecutor(max_workers=QA_WORKERS) as pool:
        list(pool.map(_tqa, qa_tasks))

qa_elapsed = time.time() - _q0
QA_GRADES = qa_ckpt.rows
qa_ok = [g for g in QA_GRADES if g.get("answer_accuracy") is not None]
print(f"\nQA: {qa_elapsed/60:.1f} min | parsed {len(qa_ok):,}/{len(QA_GRADES):,}")

## Apply drop rule and write outputs

In [ ]:
pg = defaultdict(dict)
for g in qa_ok: pg[g["pair_id"]][g["judge"]] = g

kept, dropped, review_only = [], [], []
for p in DEDUPED:
    gm = pg.get(p["id"], {})
    if len(gm) < 2:
        p["_qa_note"] = "incomplete_grades"; kept.append(p); continue
    gs = list(gm.values())
    both_contra = all(v.get("contradiction") for v in gs)
    any_acc_low = any(v.get("answer_accuracy",5) <= 2 for v in gs)
    any_qq_zero = any(v.get("question_quality",1) == 0 for v in gs)
    any_contra  = any(v.get("contradiction") for v in gs)
    if both_contra or any_acc_low or any_qq_zero:
        p["_drop_reasons"] = (
            (["both_contradiction"] if both_contra else []) +
            (["accuracy≤2"] if any_acc_low else []) +
            (["qq=0"] if any_qq_zero else []))
        dropped.append(p)
    elif any_contra:
        p["_qa_note"] = "single_contradiction_flag"
        review_only.append(p); kept.append(p)
    else:
        kept.append(p)

for p in kept:
    gm = pg.get(p["id"], {})
    for dim in ["question_quality","answer_accuracy","pair_coherence","citation_valid"]:
        vals = [v[dim] for v in gm.values() if isinstance(v.get(dim), int)]
        p[f"qa_{dim}"] = round(sum(vals)/len(vals), 2) if vals else None

print(f"Drop rule:")
print(f"  kept    : {len(kept):,} ({len(kept)/max(len(DEDUPED),1):.1%})")
print(f"  dropped : {len(dropped):,}")
print(f"  review  : {len(review_only):,} (kept but flagged)")
print(f"\nFINAL: {len(kept):,} training pairs")
print(f"  by domain     : {dict(Counter(p['domain'] for p in kept))}")
print(f"  by difficulty : {dict(Counter(p.get('difficulty') for p in kept))}")

outdir = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, run_id)
os.makedirs(outdir, exist_ok=True)

SYS = ("You are a food safety regulatory specialist with expertise in FDA FSMA regulations. "
       "Answer directly and precisely. Cite the specific CFR subsection. "
       "State conditions and exceptions. Under 150 words.")

def to_chat(p):
    return {"messages": [
        {"role": "system",    "content": SYS},
        {"role": "user",      "content": p["question"]},
        {"role": "assistant", "content": p["reference_answer"]},
    ], "metadata": {k: p.get(k)
                    for k in ["id","section","domain","difficulty","citation",
                              "batch_id","pass_idx","qa_answer_accuracy"]}}

def wj(name, rows):
    path = os.path.join(outdir, name)
    with open(path, "w") as f:
        for r in rows: f.write(json.dumps(r) + "\n")
    return path

mean_acc = round(statistics.mean(p["qa_answer_accuracy"] for p in kept
                                 if p.get("qa_answer_accuracy")), 3) if kept else 0

manifest = {
    "run_id": run_id, "stage": "4_training_pairs",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "question_run_id": rid, "generator": GENERATOR,
    "target_gross": TARGET_GROSS, "cfr_sections": total_sections,
    "cfr_chunks": total_chunks, "seed_batches": len(SEEDS),
    "raw_pairs": len(RAW), "after_dedup": len(DEDUPED),
    "dedup_eval_drop": n_ev, "dedup_internal_drop": n_int,
    "training_pairs": len(kept), "dropped": len(dropped),
    "review_flagged": len(review_only),
    "keep_rate": round(len(kept)/max(len(DEDUPED),1), 4),
    "by_domain": dict(Counter(p["domain"] for p in kept)),
    "by_difficulty": dict(Counter(p.get("difficulty") for p in kept)),
    "qa_judges": list(JUDGES.keys()), "qa_calls": len(QA_GRADES), "qa_ok": len(qa_ok),
    "mean_qa_accuracy": mean_acc,
    "generation_cost_usd": round(gen_cost, 2),
    "elapsed_min": {"generate": round(gen_elapsed/60, 1), "qa": round(qa_elapsed/60, 1)},
    "files": {
        "training_pairs":      wj("training_pairs.jsonl", kept),
        "training_pairs_chat": wj("training_pairs_chat.jsonl", [to_chat(p) for p in kept]),
        "dropped":             wj("dropped.jsonl", dropped),
        "review":              wj("review.jsonl", review_only),
        "qa_grades":           wj("qa_grades.jsonl", qa_ok),
    },
}
with open(os.path.join(outdir, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

gen_ckpt.close(); qa_ckpt.close()

print(f"\nGeneration cost  : ${manifest['generation_cost_usd']:.2f}")
print(ledger_summary())
print(f"Wall time        : {manifest['elapsed_min']['generate']:.0f}m gen + "
      f"{manifest['elapsed_min']['qa']:.0f}m QA")
print(f"Mean QA accuracy : {mean_acc}")
print(f"Output           : {outdir}")
print(f"\ntraining_pairs.jsonl      {len(kept):,} pairs")
print(f"training_pairs_chat.jsonl {len(kept):,} pairs (fine-tune format)")
print(f"review.jsonl              {len(review_only):,} pairs (check before using)")

## Inspect

In [ ]:
def show_pairs(pairs, grades, n=5, worst=True):
    gm = defaultdict(list)
    for g in grades:
        if isinstance(g.get("answer_accuracy"), int):
            gm[g["pair_id"]].append(g["answer_accuracy"])
    scored = sorted([(p, sum(gm.get(p["id"],[0]))/max(len(gm.get(p["id"],[1])),1))
                     for p in pairs], key=lambda x: x[1], reverse=not worst)
    for p, acc in scored[:n]:
        print("="*76)
        print(f"{p['id']}  21 CFR {p['section']}  domain={p['domain']}  pass={p['pass_idx']}")
        print(f"  acc={acc:.1f}  {p.get('_drop_reasons', p.get('_qa_note',''))}")
        print(f"\nQ: {p['question'][:280]}")
        print(f"A: {p['reference_answer'][:300]}\n")

if dropped:
    print(f"SAMPLE DROPPED ({min(3,len(dropped))}):\n"); show_pairs(dropped, qa_ok, n=3)
if kept:
    print(f"\nLOWEST KEPT ({min(3,len(kept))}):\n"); show_pairs(kept, qa_ok, n=3)